# Visualización de Datos Climáticos: CO₂ y Temperatura Global

Este notebook analiza datos de **emisiones de CO₂** y **anomalías de temperatura** usando tres librerías de visualización de Python. Pero en vez de repetir el mismo gráfico con cada librería, **cada una muestra su fortaleza**:

| Librería | Fortaleza | Gráfico |
|---|---|---|
| **Matplotlib** | Control total, anotaciones precisas | Línea temporal con hitos históricos |
| **Seaborn** | Estadística automática | Regresión, heatmap, distribuciones |
| **Plotly** | Interactividad y animación | Mapa animado, burbujas estilo Gapminder |

**Cómo usarlo:** ejecutá las celdas en orden (`Shift + Enter`).

## 1. Preparación

Importamos las librerías y generamos un dataset climático **sintético** basado en patrones reales (escala y tendencias de [Our World in Data](https://ourworldindata.org/co2-emissions)). Al ser sintético, no hace falta conexión a internet y el notebook es completamente autocontenido.

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
import plotly.express as px
import pandas as pd
import numpy as np

np.random.seed(42)

# =====================================================================
# Generación del dataset climático sintético
# =====================================================================

years = list(range(1960, 2024))
n_years = len(years)

# Países con datos base y waypoints de CO₂ (Mt)
paises_info = {
    "Argentina":       {"iso": "ARG", "continente": "América del Sur",
                        "pop_1960": 20.6e6,  "pop_2023": 46.6e6,
                        "co2_wp": {1960: 60, 1980: 100, 2000: 145, 2023: 180}},
    "Brasil":          {"iso": "BRA", "continente": "América del Sur",
                        "pop_1960": 72.2e6,  "pop_2023": 216e6,
                        "co2_wp": {1960: 50, 1980: 170, 2000: 340, 2023: 480}},
    "China":           {"iso": "CHN", "continente": "Asia",
                        "pop_1960": 667e6,   "pop_2023": 1412e6,
                        "co2_wp": {1960: 780, 1980: 1500, 2000: 3400, 2015: 10200, 2023: 12000}},
    "Estados Unidos":  {"iso": "USA", "continente": "América del Norte",
                        "pop_1960": 180e6,   "pop_2023": 335e6,
                        "co2_wp": {1960: 2900, 1980: 4500, 2005: 5800, 2023: 4900}},
    "India":           {"iso": "IND", "continente": "Asia",
                        "pop_1960": 450e6,   "pop_2023": 1429e6,
                        "co2_wp": {1960: 120, 1980: 350, 2000: 1000, 2023: 2900}},
    "Alemania":        {"iso": "DEU", "continente": "Europa",
                        "pop_1960": 72.8e6,  "pop_2023": 84.5e6,
                        "co2_wp": {1960: 760, 1980: 1050, 1990: 980, 2023: 600}},
    "Japón":           {"iso": "JPN", "continente": "Asia",
                        "pop_1960": 94.3e6,  "pop_2023": 125e6,
                        "co2_wp": {1960: 230, 1980: 900, 2000: 1200, 2023: 1050}},
    "Nigeria":         {"iso": "NGA", "continente": "África",
                        "pop_1960": 45.1e6,  "pop_2023": 224e6,
                        "co2_wp": {1960: 3, 1980: 15, 2000: 45, 2023: 120}},
    "Australia":       {"iso": "AUS", "continente": "Oceanía",
                        "pop_1960": 10.3e6,  "pop_2023": 26.4e6,
                        "co2_wp": {1960: 85, 1980: 200, 2000: 340, 2023: 380}},
    "Reino Unido":     {"iso": "GBR", "continente": "Europa",
                        "pop_1960": 52.4e6,  "pop_2023": 67.7e6,
                        "co2_wp": {1960: 590, 1980: 560, 2000: 530, 2023: 330}},
}

# Anomalía de temperatura global (misma para todos los países en un año)
temp_global = {
    y: -0.05 + 0.019 * (y - 1960) + np.random.normal(0, 0.08)
    for y in years
}

rows = []
for country, info in paises_info.items():
    # Interpolar CO₂ desde waypoints
    wp_years = sorted(info["co2_wp"].keys())
    wp_values = [info["co2_wp"][y] for y in wp_years]
    co2_interp = np.interp(years, wp_years, wp_values)

    for i, year in enumerate(years):
        t = i / (n_years - 1)

        # Población: interpolación lineal + ruido pequeño
        pop = info["pop_1960"] + (info["pop_2023"] - info["pop_1960"]) * t
        pop *= (1 + np.random.normal(0, 0.003))
        pop = int(pop)

        # CO₂ total + ruido + caída COVID-19
        co2 = co2_interp[i] * (1 + np.random.normal(0, 0.02))
        if year == 2020:
            co2 *= 0.93
        co2 = round(max(co2, 0.1), 1)

        # CO₂ per cápita (Mt → toneladas)
        co2_pc = round((co2 * 1e6) / pop, 2)

        # Anomalía de temperatura global
        temp = round(temp_global[year], 2)

        rows.append({
            "year": year,
            "country": country,
            "iso_code": info["iso"],
            "continent": info["continente"],
            "co2_total": co2,
            "co2_per_capita": co2_pc,
            "temperature_anomaly": temp,
            "population": pop,
        })

df = pd.DataFrame(rows)

# --- Introducir nulos realistas ---
# Nigeria: sin datos de CO₂ antes de 1971
mask = (df["country"] == "Nigeria") & (df["year"] < 1971)
df.loc[mask, ["co2_total", "co2_per_capita"]] = np.nan

# India: gaps en CO₂ per cápita en los 60s
mask = (df["country"] == "India") & (df["year"].isin([1963, 1965, 1967, 1968]))
df.loc[mask, "co2_per_capita"] = np.nan

# Argentina y Brasil: sin anomalía de temperatura antes de 1965
mask = df["country"].isin(["Argentina", "Brasil"]) & (df["year"] < 1965)
df.loc[mask, "temperature_anomaly"] = np.nan

# Australia: algunos gaps aleatorios
mask = (df["country"] == "Australia") & (df["year"].isin([1975, 1976, 1982]))
df.loc[mask, "co2_per_capita"] = np.nan

print(f"Dataset: {len(df)} filas × {df.shape[1]} columnas")
print(f"Países: {df['country'].nunique()} | Años: {df['year'].min()}–{df['year'].max()}")
df.head(10)

Cada fila es un país en un año dado. Las columnas:

| Columna | Descripción |
|---|---|
| `year` | Año (1960–2023) |
| `country` | Nombre del país |
| `iso_code` | Código ISO-3166 (necesario para mapas) |
| `continent` | Continente |
| `co2_total` | Emisiones totales de CO₂ (Mt) |
| `co2_per_capita` | Emisiones per cápita (t/persona) |
| `temperature_anomaly` | Anomalía de temperatura global (°C vs. promedio pre-industrial) |
| `population` | Población estimada |

### Calidad de los datos: nulos

Los datasets climáticos reales (como los de *Our World in Data*) tienen **muchos valores faltantes**: países que no reportaban emisiones antes de cierta fecha, estaciones meteorológicas sin cobertura, etc.

Nuestro dataset simula este problema. Veamos dónde están los huecos:

In [ ]:
print("Valores nulos por columna:\n")
print(df.isnull().sum())
print(f"\nTotal de nulos: {df.isnull().sum().sum()} de {df.size} valores "
      f"({df.isnull().sum().sum() / df.size * 100:.1f}%)")

print("\n--- Detalle por país (filas con algún nulo) ---")
nulos_por_pais = df[df.isnull().any(axis=1)].groupby("country").size()
print(nulos_por_pais)

## 2. Limpieza de datos

Antes de graficar, tratamos los nulos:
- **CO₂ total y per cápita**: interpolación lineal dentro de cada país (asumimos cambio gradual entre años conocidos).
- **Anomalía de temperatura**: no la interpolamos — los gráficos que la usen descartan las filas sin dato.

Esta estrategia es habitual en series temporales climáticas.

In [ ]:
# Interpolación por país para columnas de CO₂
for col in ["co2_total", "co2_per_capita"]:
    df[col] = df.groupby("country")[col].transform(
        lambda s: s.interpolate(method="linear", limit_direction="forward")
    )

print("Nulos restantes tras limpieza:")
print(df.isnull().sum())
print(f"\nFilas totales: {len(df)}")

### Estilo compartido

Definimos una paleta **climática** (azules fríos → rojos cálidos) y configuración base para Matplotlib/Seaborn. Los colores se eligen para que funcionen bien también en escala de grises y para personas con daltonismo.

In [ ]:
# Paleta por continente
COLORES_CONTINENTE = {
    "América del Norte": "#1b7837",
    "América del Sur":   "#41ab5d",
    "Europa":            "#4575b4",
    "Asia":              "#d73027",
    "África":            "#fc8d59",
    "Oceanía":           "#8073ac",
}

COLOR_CO2 = "#c0392b"       # rojo cálido para emisiones
COLOR_TEMP = "#2980b9"      # azul para temperatura
COLOR_NEUTRO = "#34495e"    # gris oscuro para anotaciones

ETIQUETAS = {
    "year": "Año",
    "co2_total": "Emisiones CO₂ (Mt)",
    "co2_per_capita": "CO₂ per cápita (t/persona)",
    "temperature_anomaly": "Anomalía de temperatura (°C)",
    "population": "Población",
    "continent": "Continente",
    "country": "País",
}

# Estilo base para Matplotlib y Seaborn
plt.rcParams.update({
    "figure.figsize": (10, 5.5),
    "axes.spines.top": False,
    "axes.spines.right": False,
    "axes.edgecolor": "#c3c2b7",
    "axes.grid": True,
    "axes.axisbelow": True,
    "grid.color": "#e8e7e3",
    "grid.linewidth": 0.6,
    "axes.titlesize": 14,
    "axes.titleweight": "bold",
    "font.size": 11,
})

---
## 3. 🎨 Matplotlib — Control total: líneas con anotaciones

Matplotlib brilla cuando necesitás **control pixel a pixel** sobre el gráfico. Acá creamos un gráfico de líneas con:

- **Doble eje Y** (`twinx`): CO₂ a la izquierda, temperatura a la derecha
- **Anotaciones con flechas** en hitos históricos (Protocolo de Kyoto, Acuerdo de París, COVID-19)
- **Sombreado** para destacar períodos clave

Este tipo de personalización es muy difícil (o imposible) con Seaborn o Plotly.

In [ ]:
# Datos globales: suma de CO₂ de todos los países, promedio de temperatura
global_data = df.groupby("year").agg(
    co2_total=("co2_total", "sum"),
    temp_anomaly=("temperature_anomaly", "mean"),
).reset_index()

fig, ax1 = plt.subplots(figsize=(12, 6))

# --- Eje izquierdo: CO₂ total ---
ax1.fill_between(global_data["year"], global_data["co2_total"],
                 alpha=0.15, color=COLOR_CO2)
ax1.plot(global_data["year"], global_data["co2_total"],
         color=COLOR_CO2, linewidth=2.5, label="CO₂ total (Mt)")
ax1.set_xlabel("Año", fontsize=12)
ax1.set_ylabel("Emisiones globales de CO₂ (Mt)", color=COLOR_CO2, fontsize=12)
ax1.tick_params(axis="y", labelcolor=COLOR_CO2)
ax1.yaxis.set_major_formatter(
    mticker.FuncFormatter(lambda x, _: f"{x/1000:.0f}k")
)

# --- Eje derecho: anomalía de temperatura ---
ax2 = ax1.twinx()
# Restaurar el spine derecho que ocultamos en rcParams
ax2.spines["right"].set_visible(True)
ax2.spines["right"].set_color(COLOR_TEMP)
temp_clean = global_data.dropna(subset=["temp_anomaly"])
ax2.plot(temp_clean["year"], temp_clean["temp_anomaly"],
         color=COLOR_TEMP, linewidth=2, linestyle="--", alpha=0.9,
         label="Anomalía temp. (°C)")
ax2.set_ylabel("Anomalía de temperatura (°C)", color=COLOR_TEMP, fontsize=12)
ax2.tick_params(axis="y", labelcolor=COLOR_TEMP)

# --- Anotaciones en hitos históricos ---
hitos = [
    (1997, "Protocolo\nde Kyoto",  -60),
    (2015, "Acuerdo\nde París",    -70),
    (2020, "COVID-19",               50),
]

for year_h, texto, offset_y in hitos:
    co2_val = global_data.loc[global_data["year"] == year_h, "co2_total"].values[0]
    ax1.annotate(
        texto,
        xy=(year_h, co2_val),
        xytext=(0, offset_y),
        textcoords="offset points",
        fontsize=9, fontweight="bold", color=COLOR_NEUTRO, ha="center",
        arrowprops=dict(arrowstyle="->", color=COLOR_NEUTRO, lw=1.5),
        bbox=dict(boxstyle="round,pad=0.3", facecolor="white",
                  edgecolor=COLOR_NEUTRO, alpha=0.9),
    )

# Sombreado: período de aceleración industrial china (2000–2013)
ax1.axvspan(2000, 2013, alpha=0.07, color="#e67e22",
            label="Aceleración industrial China")

# --- Leyenda combinada ---
lines1, labels1 = ax1.get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()
ax1.legend(lines1 + lines2, labels1 + labels2,
           loc="upper left", fontsize=9, framealpha=0.9)

ax1.set_title(
    "Emisiones globales de CO₂ y anomalía de temperatura (1960–2023)",
    fontsize=14, fontweight="bold", pad=15,
)
plt.tight_layout()
plt.show()

**~40 líneas de código** para un gráfico con doble eje, anotaciones, sombreado y leyenda combinada. Matplotlib exige escribir mucho, pero el resultado es un gráfico de **calidad publicación** con control total sobre cada elemento.

---
## 4. 📈 Seaborn — Estadística automática: tendencias y distribuciones

Seaborn brilla en **análisis estadístico visual**: regresiones, intervalos de confianza, distribuciones y facetados. Hace en **una línea** lo que Matplotlib necesita 20.

### 4.1 Regresión con facetado por continente

`lmplot` ajusta una recta de regresión **automáticamente** (con banda de confianza al 95%) y con `col="continent"` crea un panel por cada continente. Esto muestra cómo evolucionó el CO₂ per cápita en cada región del mundo.

In [ ]:
# Submuestreo: un dato cada 3 años para que no quede saturado
df_sample = df[df["year"].isin(range(1960, 2024, 3))].dropna(
    subset=["co2_per_capita"]
)

g = sns.lmplot(
    data=df_sample,
    x="year",
    y="co2_per_capita",
    col="continent",
    col_wrap=3,
    hue="continent",
    palette=COLORES_CONTINENTE,
    height=3.5,
    aspect=1.2,
    scatter_kws={"s": 25, "alpha": 0.6},
    line_kws={"linewidth": 2},
    ci=95,               # banda de confianza automática
    legend=False,
)

g.set_titles("{col_name}", fontweight="bold")
g.set_axis_labels("Año", "CO₂ per cápita (t/persona)")
g.figure.suptitle(
    "Tendencia de CO₂ per cápita por continente "
    "(con intervalo de confianza al 95%)",
    fontsize=14, fontweight="bold", y=1.02,
)
plt.tight_layout()
plt.show()

### 4.2 Mapa de calor: correlaciones

`heatmap` genera una **matriz de correlación** con un solo llamado. Seaborn elige automáticamente la escala de colores y las anotaciones numéricas. Útil para detectar rápidamente qué variables están relacionadas.

In [ ]:
cols_num = ["co2_total", "co2_per_capita", "temperature_anomaly",
            "population", "year"]
corr = df[cols_num].dropna().corr()

# Renombrar para el gráfico
nombres = {
    "co2_total": "CO₂ total",
    "co2_per_capita": "CO₂/cápita",
    "temperature_anomaly": "Anom. temp.",
    "population": "Población",
    "year": "Año",
}
corr = corr.rename(index=nombres, columns=nombres)

fig, ax = plt.subplots(figsize=(7, 6))
sns.heatmap(
    corr,
    annot=True,             # escribe el valor en cada celda
    fmt=".2f",
    cmap="RdBu_r",          # azul (negativo) → rojo (positivo)
    center=0,
    vmin=-1, vmax=1,
    square=True,
    linewidths=1,
    cbar_kws={"shrink": 0.8, "label": "Correlación de Pearson"},
    ax=ax,
)
ax.set_title("Matriz de correlación entre variables climáticas",
             fontsize=14, fontweight="bold", pad=15)
plt.tight_layout()
plt.show()

### 4.3 Violinplot: distribución de CO₂ per cápita por continente

El violinplot muestra la **forma completa de la distribución** (no solo los cuartiles como un boxplot). Seaborn además dibuja un mini-boxplot adentro con `inner="box"`.

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5.5))

sns.violinplot(
    data=df.dropna(subset=["co2_per_capita"]),
    x="continent",
    y="co2_per_capita",
    hue="continent",
    palette=COLORES_CONTINENTE,
    inner="box",
    linewidth=1.2,
    saturation=0.85,
    density_norm="width",
    legend=False,
    ax=ax,
)

ax.set_xlabel("Continente", fontsize=12)
ax.set_ylabel("CO₂ per cápita (t/persona)", fontsize=12)
ax.set_title("Distribución de CO₂ per cápita por continente (1960–2023)",
             fontsize=14, fontweight="bold")
ax.tick_params(axis="x", rotation=15)
plt.tight_layout()
plt.show()

**3 gráficos estadísticos en ~15 líneas cada uno.** Seaborn calculó automáticamente regresiones, bandas de confianza, correlaciones y densidades. En Matplotlib, cada uno de esos cálculos habría que programarlo a mano.

---
## 5. 🌍 Plotly — Interactividad: mapas animados y exploración

Plotly brilla en **interactividad y animación**. Un mapa coroplético animado o un gráfico de burbujas estilo Gapminder son casi imposibles de hacer con Matplotlib o Seaborn.

### 5.1 Mapa coroplético animado

Con `px.choropleth` y `animation_frame="year"`, Plotly genera un mapa mundial donde podés:

- **Reproducir la animación** para ver cómo cambian las emisiones por país
- **Pasar el mouse** sobre un país para ver su valor exacto
- **Pausar, avanzar o retroceder** con el slider

Todo esto en **~10 líneas de código**.

In [ ]:
fig = px.choropleth(
    df.dropna(subset=["co2_per_capita"]),
    locations="iso_code",
    color="co2_per_capita",
    hover_name="country",
    animation_frame="year",
    color_continuous_scale="YlOrRd",
    range_color=[0, df["co2_per_capita"].quantile(0.95)],
    labels=ETIQUETAS,
    title="Emisiones de CO₂ per cápita por país (1960–2023)",
)

fig.update_layout(
    geo=dict(showframe=False, showcoastlines=True,
             projection_type="natural earth"),
    coloraxis_colorbar=dict(title="t CO₂/persona"),
    width=900, height=550,
)
fig.show()

### 5.2 Gráfico de burbujas animado (estilo Gapminder)

El famoso formato de [Hans Rosling](https://www.gapminder.org/): cada burbuja es un **país**, el tamaño representa la **población**, el color el **continente**, y la animación avanza por **año**.

Plotly lo hace con `px.scatter` + `animation_frame`.

In [ ]:
df_bubble = df.dropna(subset=["co2_per_capita", "temperature_anomaly"])

fig = px.scatter(
    df_bubble,
    x="co2_per_capita",
    y="temperature_anomaly",
    size="population",
    color="continent",
    hover_name="country",
    animation_frame="year",
    size_max=55,
    color_discrete_map=COLORES_CONTINENTE,
    labels=ETIQUETAS,
    title="CO₂ per cápita vs. Anomalía de temperatura — Evolución por año",
    range_x=[0, df_bubble["co2_per_capita"].quantile(0.99) * 1.1],
    range_y=[df_bubble["temperature_anomaly"].min() - 0.2,
             df_bubble["temperature_anomaly"].max() + 0.2],
)

fig.update_layout(
    width=900, height=600,
    legend=dict(title="Continente", font=dict(size=11)),
)
fig.show()

**~12 líneas de código por gráfico** para obtener mapas animados y gráficos interactivos. Plotly genera HTML/JavaScript por detrás, lo que permite interactividad que es **imposible** con librerías estáticas como Matplotlib o Seaborn.

---
## 6. Conclusiones

| | Matplotlib | Seaborn | Plotly |
|---|---|---|---|
| **Fortaleza** | Control total, anotaciones, personalización | Estadística automática, facetados, distribuciones | Interactividad, mapas, animaciones |
| **Gráfico demo** | Líneas con anotaciones históricas y doble eje | Regresión + heatmap + violinplot | Mapa animado + burbujas Gapminder |
| **Código necesario** | Mucho (~40 líneas) | Poco (~15 líneas) | Poco (~12 líneas) |
| **Resultado** | Estático (imagen), calidad publicación | Estático, análisis rápido | Interactivo (HTML/JS) |
| **Mejor para** | Papers, informes finales, gráficos a medida | Exploración estadística, notebooks de análisis | Dashboards, presentaciones, exploración de datos |
| **Curva de aprendizaje** | Alta — hay que conocer muchos métodos | Media — la API es consistente | Baja — `px.algo()` y listo |

### ¿Cuál usar?

No es una cuestión de "mejor o peor". **Las tres se complementan**:

1. Empezá la exploración con **Seaborn** (rápido, estadístico).
2. Si necesitás interactividad o presentar a un público no técnico → **Plotly**.
3. Si necesitás un gráfico perfecto para una publicación → **Matplotlib**.

En la práctica, muchos proyectos usan las tres en distintas etapas del análisis.